<a href="https://colab.research.google.com/github/chinmay227/indian-market-internals/blob/main/notebooks/04_build_research_universe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build the Research Universe

## Objective

Construct a reproducible 100-stock research universe sampled from the current Nifty 500.

## Inputs

- Current Nifty 500 constituents
- NSE sector classification
- Market-cap information for ranking constituents
- Official Nifty sector-index mappings

## Method

1. Start from the current Nifty 500 constituent universe.
2. Rank constituents by market capitalization.
3. Divide the 500 stocks into five market-cap strata:
   - ranks 1–100
   - ranks 101–200
   - ranks 201–300
   - ranks 301–400
   - ranks 401–500
4. Select 20 stocks from each stratum.
5. Make the random selection sector-aware so that sector representation is reasonably preserved.
6. Use a fixed random seed so the same 100 stocks are selected every time.
7. Map each selected stock to an appropriate official Nifty sector benchmark where one exists.

## Output

A reproducible 100-stock research-universe table containing:

- company name
- ticker
- market-cap rank
- market-cap stratum
- NSE sector
- industry
- sector benchmark
- sector benchmark availability

## Limitations

This first version uses current Nifty 500 membership for historical analysis. Therefore, it does not eliminate survivorship bias or reconstruct historical index membership.

The universe is intentionally size-stratified and is not intended to represent the Nifty 500 in proportion to market capitalization.

In [2]:
import pandas as pd
import requests
from io import BytesIO

NIFTY500_URL = (
    "https://www.niftyindices.com/"
    "IndexConstituent/ind_nifty500list.csv"
)

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    NIFTY500_URL,
    headers=headers,
    timeout=20
)

response.raise_for_status()

nifty500 = pd.read_csv(BytesIO(response.content))

print("Number of constituents:", len(nifty500))
print("\nColumns:")
print(nifty500.columns.tolist())

nifty500.head()

Number of constituents: 500

Columns:
['Company Name', 'Industry', 'Symbol', 'Series', 'ISIN Code']


,Company Name,Industry,Symbol,Series,ISIN Code
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038
1,3M India Ltd.,Diversified,3MINDIA,EQ,INE470A01017
2,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022
3,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025
4,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025


In [3]:
industry_values = sorted(
    nifty500["Industry"]
    .dropna()
    .unique()
)

print("Number of unique classifications:", len(industry_values))

for value in industry_values:
    print(value)

Number of unique classifications: 20
Automobile and Auto Components
Capital Goods
Chemicals
Construction
Construction Materials
Consumer Durables
Consumer Services
Diversified
Fast Moving Consumer Goods
Financial Services
Healthcare
Information Technology
Media Entertainment & Publication
Metals & Mining
Oil Gas & Consumable Fuels
Power
Realty
Services
Telecommunication
Textiles


In [4]:
research_universe = (
    nifty500[
        ["Company Name", "Symbol", "Industry", "ISIN Code"]
    ]
    .rename(
        columns={
            "Company Name": "company_name",
            "Symbol": "ticker",
            "Industry": "sector",
            "ISIN Code": "isin",
        }
    )
    .copy()
)

research_universe.head()

,company_name,ticker,sector,isin
0,360 ONE WAM Ltd.,360ONE,Financial Services,INE466L01038
1,3M India Ltd.,3MINDIA,Diversified,INE470A01017
2,ABB India Ltd.,ABB,Capital Goods,INE117A01022
3,ACC Ltd.,ACC,Construction Materials,INE012A01025
4,ACME Solar Holdings Ltd.,ACMESOLAR,Power,INE622W01025


In [5]:
research_universe["sector"].value_counts()

,count
sector,
Financial Services,101
Capital Goods,63
Healthcare,48
Automobile and Auto Components,38
Consumer Services,29
Fast Moving Consumer Goods,28
Information Technology,27
Chemicals,26
Metals & Mining,18


In [6]:
from google.colab import files

uploaded = files.upload()

market_cap_filename = next(iter(uploaded))

print("Uploaded file:", market_cap_filename)

Saving Average_MCAP_July2025ToDecember2025_20260102201101.xlsx to Average_MCAP_July2025ToDecember2025_20260102201101 (1).xlsx
Uploaded file: Average_MCAP_July2025ToDecember2025_20260102201101 (1).xlsx


In [7]:
market_cap = pd.read_excel(market_cap_filename)

print("Rows in market-cap file:", len(market_cap))
print("\nColumns:")
print(market_cap.columns.tolist())

market_cap.head()

Rows in market-cap file: 2869

Columns:
['Rank', 'Symbol', 'Company Name', 'Average market capitalisation from July  01, 2025 to December 31, 2025 (Rs. In lakhs)']


,Rank,Symbol,Company Name,"Average market capitalisation from July 01, 2025 to December 31, 2025 (Rs. In lakhs)"
0,1,RELIANCE,Reliance Industries Limited,197078768.997765
1,2,HDFCBANK,HDFC Bank Limited,151558090.449832
2,3,BHARTIARTL,Bharti Airtel Limited,113625128.614847
3,4,TCS,Tata Consultancy Services Limited,112955514.997768
4,5,ICICIBANK,ICICI Bank Limited,99863063.768613


In [10]:
market_cap_clean = (
    market_cap[
        [
            "Symbol",
            "Average market capitalisation from July  01, 2025 to December 31, 2025 (Rs. In lakhs)"
        ]
    ]
    .rename(
        columns={
            "Symbol": "ticker",
            "Average market capitalisation from July  01, 2025 to December 31, 2025 (Rs. In lakhs)": "market_cap_lakh"
        }
    )
    .dropna(subset=["ticker"])
    .copy()
)

print("Clean market-cap rows:", len(market_cap_clean))
print(
    "Duplicate tickers:",
    market_cap_clean["ticker"].duplicated().sum()
)

market_cap_clean.head()

Clean market-cap rows: 2867
Duplicate tickers: 0


,ticker,market_cap_lakh
0,RELIANCE,197078768.997765
1,HDFCBANK,151558090.449832
2,BHARTIARTL,113625128.614847
3,TCS,112955514.997768
4,ICICIBANK,99863063.768613


In [11]:
research_universe = research_universe.merge(
    market_cap_clean,
    on="ticker",
    how="left",
    validate="one_to_one"
)

print(
    "Stocks missing market cap:",
    research_universe["market_cap_lakh"].isna().sum()
)

research_universe.head()

Stocks missing market cap: 5


,company_name,ticker,sector,isin,market_cap_lakh
0,360 ONE WAM Ltd.,360ONE,Financial Services,INE466L01038,4438113.093452
1,3M India Ltd.,3MINDIA,Diversified,INE470A01017,3578219.84779
2,ABB India Ltd.,ABB,Capital Goods,INE117A01022,11155790.265937
3,ACC Ltd.,ACC,Construction Materials,INE012A01025,3469849.584708
4,ACME Solar Holdings Ltd.,ACMESOLAR,Power,INE622W01025,1626138.209201


In [12]:
missing_market_cap = research_universe[
    research_universe["market_cap_lakh"].isna()
][
    ["company_name", "ticker", "sector", "isin"]
]

missing_market_cap

,company_name,ticker,sector,isin
12,Abbott India Ltd.,ABBOTINDIA,Healthcare,INE358A01014
72,Bayer Cropscience Ltd.,BAYERCROP,Chemicals,INE462A01022
260,JSW Dulux Ltd.,JSWDULUX,Consumer Durables,INE133A01011
294,LTM Ltd.,LTM,Information Technology,INE214T01019
324,Multi Commodity Exchange of India Ltd.,MCX,Financial Services,INE745G01043


In [13]:
print("Missing stocks:", len(missing_market_cap))

for ticker in missing_market_cap["ticker"]:
    print(ticker)

Missing stocks: 5
ABBOTINDIA
BAYERCROP
JSWDULUX
LTM
MCX


In [14]:
market_cap_aliases = {
    "JSWDULUX": "AKZOINDIA",
    "LTM": "LTIM",
}

market_cap_lookup = research_universe["ticker"].replace(
    market_cap_aliases
)

market_cap_by_ticker = (
    market_cap_clean
    .set_index("ticker")["market_cap_lakh"]
)

research_universe["market_cap_lakh"] = (
    market_cap_lookup.map(market_cap_by_ticker)
)

missing_market_cap = research_universe[
    research_universe["market_cap_lakh"].isna()
][
    ["company_name", "ticker", "sector", "isin"]
]

print(
    "Stocks still missing market cap:",
    len(missing_market_cap)
)

missing_market_cap

Stocks still missing market cap: 3


,company_name,ticker,sector,isin
12,Abbott India Ltd.,ABBOTINDIA,Healthcare,INE358A01014
72,Bayer Cropscience Ltd.,BAYERCROP,Chemicals,INE462A01022
324,Multi Commodity Exchange of India Ltd.,MCX,Financial Services,INE745G01043


In [15]:
fallback_market_caps_crore = {
    "ABBOTINDIA": 65871.14,
    "MCX": 45092.61,
    "BAYERCROP": 23405.20,
}

fallback_market_caps_lakh = {
    ticker: value * 100
    for ticker, value in fallback_market_caps_crore.items()
}

missing_mask = research_universe["market_cap_lakh"].isna()

research_universe.loc[
    missing_mask,
    "market_cap_lakh"
] = (
    research_universe.loc[missing_mask, "ticker"]
    .map(fallback_market_caps_lakh)
)

print(
    "Stocks still missing market cap:",
    research_universe["market_cap_lakh"].isna().sum()
)

Stocks still missing market cap: 0


In [16]:
research_universe["market_cap_source"] = (
    "NSE 6M average market cap, Jul-Dec 2025"
)

research_universe.loc[
    research_universe["ticker"].isin(fallback_market_caps_crore),
    "market_cap_source"
] = "AMFI 6M average market cap, Jul-Dec 2025"

In [17]:
research_universe = (
    research_universe
    .sort_values(
        ["market_cap_lakh", "ticker"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

research_universe["market_cap_rank"] = (
    research_universe.index + 1
)

research_universe["market_cap_stratum"] = (
    (research_universe["market_cap_rank"] - 1) // 100 + 1
)

research_universe.head(10)

,company_name,ticker,sector,isin,market_cap_lakh,market_cap_source,market_cap_rank,market_cap_stratum
0,Reliance Industries Ltd.,RELIANCE,Oil Gas & Consumable Fuels,INE002A01018,197078768.997765,"NSE 6M average market cap, Jul-Dec 2025",1,1
1,HDFC Bank Ltd.,HDFCBANK,Financial Services,INE040A01034,151558090.449832,"NSE 6M average market cap, Jul-Dec 2025",2,1
2,Bharti Airtel Ltd.,BHARTIARTL,Telecommunication,INE397D01024,113625128.614847,"NSE 6M average market cap, Jul-Dec 2025",3,1
3,Tata Consultancy Services Ltd.,TCS,Information Technology,INE467B01029,112955514.997768,"NSE 6M average market cap, Jul-Dec 2025",4,1
4,ICICI Bank Ltd.,ICICIBANK,Financial Services,INE090A01021,99863063.768613,"NSE 6M average market cap, Jul-Dec 2025",5,1
5,State Bank of India,SBIN,Financial Services,INE062A01020,80055785.311131,"NSE 6M average market cap, Jul-Dec 2025",6,1
6,Infosys Ltd.,INFY,Information Technology,INE009A01021,63510354.939534,"NSE 6M average market cap, Jul-Dec 2025",7,1
7,Bajaj Finance Ltd.,BAJFINANCE,Financial Services,INE296A01032,60799294.450156,"NSE 6M average market cap, Jul-Dec 2025",8,1
8,Hindustan Unilever Ltd.,HINDUNILVR,Fast Moving Consumer Goods,INE030A01027,58251812.469492,"NSE 6M average market cap, Jul-Dec 2025",9,1
9,Life Insurance Corporation of India,LICI,Financial Services,INE0J1Y01017,56525902.073104,"NSE 6M average market cap, Jul-Dec 2025",10,1


In [18]:
research_universe[
    "market_cap_stratum"
].value_counts().sort_index()

,count
market_cap_stratum,
1,100
2,100
3,100
4,100
5,100


In [19]:
research_universe.loc[
    [98, 99, 100, 198, 199, 200, 298, 299, 300, 398, 399, 400],
    [
        "market_cap_rank",
        "company_name",
        "ticker",
        "sector",
        "market_cap_lakh",
        "market_cap_stratum"
    ]
]

,market_cap_rank,company_name,ticker,sector,market_cap_lakh,market_cap_stratum
98,99,Hero MotoCorp Ltd.,HEROMOTOCO,Automobile and Auto Components,10554351.319525,1
99,100,Dr. Reddy's Laboratories Ltd.,DRREDDY,Healthcare,10517030.361196,1
100,101,Shree Cement Ltd.,SHREECEM,Construction Materials,10452277.704943,2
198,199,Laurus Labs Ltd.,LAURUSLABS,Healthcare,4935441.284854,2
199,200,Authum Investment & Infrastructure Ltd.,AIIL,Financial Services,4931828.564079,2
200,201,Godfrey Phillips India Ltd.,GODFRYPHLP,Fast Moving Consumer Goods,4905377.523954,3
298,299,Firstsource Solutions Ltd.,FSL,Services,2440870.170162,3
299,300,Emami Ltd.,EMAMILTD,Fast Moving Consumer Goods,2440461.107143,3
300,301,Deepak Nitrite Ltd.,DEEPAKNTR,Chemicals,2420073.685717,4
398,399,Allied Blenders and Distillers Ltd.,ABDL,Fast Moving Consumer Goods,1552334.730043,4


In [20]:
sector_by_stratum = pd.crosstab(
    research_universe["sector"],
    research_universe["market_cap_stratum"]
)

sector_by_stratum["Total"] = sector_by_stratum.sum(axis=1)

sector_by_stratum

market_cap_stratum,1,2,3,4,5,Total
sector,,,,,,
Automobile and Auto Components,10,5,7,9,7,38
Capital Goods,10,8,13,13,19,63
Chemicals,2,6,3,7,8,26
Construction,1,1,2,5,4,13
Construction Materials,3,1,4,1,2,11
Consumer Durables,3,4,3,5,1,16
Consumer Services,4,7,3,8,7,29
Diversified,0,0,2,1,0,3
Fast Moving Consumer Goods,7,5,6,3,7,28


In [21]:
sector_by_stratum.loc["Total"] = sector_by_stratum.sum(axis=0)

sector_by_stratum

market_cap_stratum,1,2,3,4,5,Total
sector,,,,,,
Automobile and Auto Components,10,5,7,9,7,38
Capital Goods,10,8,13,13,19,63
Chemicals,2,6,3,7,8,26
Construction,1,1,2,5,4,13
Construction Materials,3,1,4,1,2,11
Consumer Durables,3,4,3,5,1,16
Consumer Services,4,7,3,8,7,29
Diversified,0,0,2,1,0,3
Fast Moving Consumer Goods,7,5,6,3,7,28


In [22]:
sector_sample_targets = (
    research_universe["sector"]
    .value_counts()
    .rename("available_stocks")
    .to_frame()
)

# Start from the natural target: 5 stocks per sector
sector_sample_targets["target_sample"] = 5

# Small-sector adjustments
sector_sample_targets.loc[
    "Diversified",
    "target_sample"
] = 3

sector_sample_targets.loc[
    [
        "Media Entertainment & Publication",
        "Textiles"
    ],
    "target_sample"
] = 4

# How many seats remain?
remaining_slots = (
    100
    - sector_sample_targets["target_sample"].sum()
)

print("Remaining slots:", remaining_slots)

Remaining slots: 4


In [23]:
eligible_for_extra = (
    sector_sample_targets[
        sector_sample_targets["target_sample"] < 6
    ]
    .sort_values(
        "available_stocks",
        ascending=False
    )
)

extra_sectors = eligible_for_extra.head(
    remaining_slots
).index

sector_sample_targets.loc[
    extra_sectors,
    "target_sample"
] += 1

print("Total target sample:", sector_sample_targets["target_sample"].sum())

sector_sample_targets.sort_values(
    ["target_sample", "available_stocks"],
    ascending=[False, False]
)

Total target sample: 100


,available_stocks,target_sample
sector,,
Financial Services,101,6
Capital Goods,63,6
Healthcare,48,6
Automobile and Auto Components,38,6
Consumer Services,29,5
Fast Moving Consumer Goods,28,5
Information Technology,27,5
Chemicals,26,5
Metals & Mining,18,5


In [24]:
import numpy as np

sectors = sector_sample_targets.index.tolist()
strata = [1, 2, 3, 4, 5]

availability = (
    sector_by_stratum
    .drop(index="Total", errors="ignore")
    .drop(columns="Total", errors="ignore")
    .loc[sectors, strata]
)

targets = (
    sector_sample_targets
    .loc[sectors, "target_sample"]
)

availability

market_cap_stratum,1,2,3,4,5
sector,,,,,
Financial Services,24,27,21,19,10
Capital Goods,10,8,13,13,19
Healthcare,7,10,14,11,6
Automobile and Auto Components,10,5,7,9,7
Consumer Services,4,7,3,8,7
Fast Moving Consumer Goods,7,5,6,3,7
Information Technology,6,4,7,3,7
Chemicals,2,6,3,7,8
Metals & Mining,6,5,2,2,3


In [25]:
from scipy.optimize import milp, LinearConstraint, Bounds

n_sectors = len(sectors)
n_strata = len(strata)

# One decision variable for every sector × stratum combination
n_x = n_sectors * n_strata

# We also create deviation variables so the optimizer can
# prefer spreading each sector across strata.
n_variables = n_x * 2

# Objective:
# x variables have zero cost
# deviation variables have cost 1
c = np.concatenate([
    np.zeros(n_x),
    np.ones(n_x)
])

# x variables must be integers.
# deviation variables may be continuous.
integrality = np.concatenate([
    np.ones(n_x),
    np.zeros(n_x)
])

lower_bounds = np.zeros(n_variables)

upper_bounds = np.concatenate([
    availability.to_numpy().flatten(),
    np.full(n_x, np.inf)
])

bounds = Bounds(lower_bounds, upper_bounds)

In [26]:
constraint_rows = []
lower_limits = []
upper_limits = []

# 1. Each sector must hit its target sample size
for sector_idx, sector in enumerate(sectors):
    row = np.zeros(n_variables)

    start = sector_idx * n_strata
    row[start:start + n_strata] = 1

    constraint_rows.append(row)

    target = targets.loc[sector]
    lower_limits.append(target)
    upper_limits.append(target)


# 2. Each market-cap stratum must contain exactly 20 stocks
for stratum_idx in range(n_strata):
    row = np.zeros(n_variables)

    for sector_idx in range(n_sectors):
        x_position = sector_idx * n_strata + stratum_idx
        row[x_position] = 1

    constraint_rows.append(row)
    lower_limits.append(20)
    upper_limits.append(20)

In [27]:
# 3. Prefer each sector to be spread reasonably evenly
# across the five market-cap strata.

for sector_idx, sector in enumerate(sectors):

    ideal_per_stratum = targets.loc[sector] / n_strata

    for stratum_idx in range(n_strata):

        x_position = sector_idx * n_strata + stratum_idx
        deviation_position = n_x + x_position

        # x - deviation <= ideal
        row = np.zeros(n_variables)
        row[x_position] = 1
        row[deviation_position] = -1

        constraint_rows.append(row)
        lower_limits.append(-np.inf)
        upper_limits.append(ideal_per_stratum)

        # x + deviation >= ideal
        row = np.zeros(n_variables)
        row[x_position] = 1
        row[deviation_position] = 1

        constraint_rows.append(row)
        lower_limits.append(ideal_per_stratum)
        upper_limits.append(np.inf)

In [28]:
constraints = LinearConstraint(
    np.array(constraint_rows),
    np.array(lower_limits),
    np.array(upper_limits)
)

result = milp(
    c=c,
    integrality=integrality,
    bounds=bounds,
    constraints=constraints
)

print("Optimization successful:", result.success)
print("Status:", result.message)

Optimization successful: True
Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [29]:
allocation_values = np.rint(
    result.x[:n_x]
).astype(int)

sector_stratum_allocation = pd.DataFrame(
    allocation_values.reshape(n_sectors, n_strata),
    index=sectors,
    columns=strata
)

sector_stratum_allocation["Total"] = (
    sector_stratum_allocation.sum(axis=1)
)

sector_stratum_allocation

,1,2,3,4,5,Total
Financial Services,1,1,1,2,1,6
Capital Goods,2,1,1,2,0,6
Healthcare,2,1,1,1,1,6
Automobile and Auto Components,2,1,1,1,1,6
Consumer Services,1,1,1,1,1,5
Fast Moving Consumer Goods,1,1,1,1,1,5
Information Technology,1,1,1,1,1,5
Chemicals,1,1,1,1,1,5
Metals & Mining,1,1,1,1,1,5
Oil Gas & Consumable Fuels,1,1,1,1,1,5


In [30]:
print("Stocks allocated per market-cap stratum:")
print(
    sector_stratum_allocation[strata].sum(axis=0)
)

print("\nSector targets matched:")
print(
    (
        sector_stratum_allocation["Total"]
        == targets
    ).all()
)

print("\nAllocation exceeds availability anywhere:")
print(
    (
        sector_stratum_allocation[strata]
        > availability
    ).any().any()
)

Stocks allocated per market-cap stratum:
1    20
2    20
3    20
4    20
5    20
dtype: int64

Sector targets matched:
True

Allocation exceeds availability anywhere:
False


In [31]:
sector_stratum_allocation.loc["Total"] = (
    sector_stratum_allocation.sum(axis=0)
)

sector_stratum_allocation

,1,2,3,4,5,Total
Financial Services,1,1,1,2,1,6
Capital Goods,2,1,1,2,0,6
Healthcare,2,1,1,1,1,6
Automobile and Auto Components,2,1,1,1,1,6
Consumer Services,1,1,1,1,1,5
Fast Moving Consumer Goods,1,1,1,1,1,5
Information Technology,1,1,1,1,1,5
Chemicals,1,1,1,1,1,5
Metals & Mining,1,1,1,1,1,5
Oil Gas & Consumable Fuels,1,1,1,1,1,5


In [32]:
RANDOM_SEED = 42

In [33]:
sampled_groups = []

for sector in sectors:
    for stratum in strata:

        n_to_sample = sector_stratum_allocation.loc[
            sector,
            stratum
        ]

        if n_to_sample == 0:
            continue

        eligible = research_universe[
            (research_universe["sector"] == sector)
            & (
                research_universe["market_cap_stratum"]
                == stratum
            )
        ]

        sampled = eligible.sample(
            n=n_to_sample,
            random_state=RANDOM_SEED
        )

        sampled_groups.append(sampled)

sampled_universe = (
    pd.concat(sampled_groups)
    .sort_values("market_cap_rank")
    .reset_index(drop=True)
)

print("Total sampled stocks:", len(sampled_universe))

sampled_universe.head(10)

Total sampled stocks: 100


,company_name,ticker,sector,isin,market_cap_lakh,market_cap_source,market_cap_rank,market_cap_stratum
0,Reliance Industries Ltd.,RELIANCE,Oil Gas & Consumable Fuels,INE002A01018,197078768.997765,"NSE 6M average market cap, Jul-Dec 2025",1,1
1,Bharti Airtel Ltd.,BHARTIARTL,Telecommunication,INE397D01024,113625128.614847,"NSE 6M average market cap, Jul-Dec 2025",3,1
2,Tata Consultancy Services Ltd.,TCS,Information Technology,INE467B01029,112955514.997768,"NSE 6M average market cap, Jul-Dec 2025",4,1
3,Hindustan Unilever Ltd.,HINDUNILVR,Fast Moving Consumer Goods,INE030A01027,58251812.469492,"NSE 6M average market cap, Jul-Dec 2025",9,1
4,Larsen & Toubro Ltd.,LT,Construction,INE018A01030,51876327.592404,"NSE 6M average market cap, Jul-Dec 2025",11,1
5,Mahindra & Mahindra Ltd.,M&M,Automobile and Auto Components,INE101A01026,43188595.189127,"NSE 6M average market cap, Jul-Dec 2025",14,1
6,Sun Pharmaceutical Industries Ltd.,SUNPHARMA,Healthcare,INE044A01036,40440695.707645,"NSE 6M average market cap, Jul-Dec 2025",17,1
7,UltraTech Cement Ltd.,ULTRACEMCO,Construction Materials,INE481G01011,35804548.314875,"NSE 6M average market cap, Jul-Dec 2025",19,1
8,Titan Company Ltd.,TITAN,Consumer Durables,INE280A01028,32361306.313366,"NSE 6M average market cap, Jul-Dec 2025",21,1
9,NTPC Ltd.,NTPC,Power,INE733E01010,32323374.811683,"NSE 6M average market cap, Jul-Dec 2025",22,1


In [34]:
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

sampled_groups = []

for sector in sectors:
    for stratum in strata:

        n_to_sample = sector_stratum_allocation.loc[
            sector,
            stratum
        ]

        if n_to_sample == 0:
            continue

        eligible = research_universe[
            (research_universe["sector"] == sector)
            & (
                research_universe["market_cap_stratum"]
                == stratum
            )
        ]

        group_seed = int(
            rng.integers(0, 1_000_000)
        )

        sampled = eligible.sample(
            n=n_to_sample,
            random_state=group_seed
        )

        sampled_groups.append(sampled)

sampled_universe = (
    pd.concat(sampled_groups)
    .sort_values("market_cap_rank")
    .reset_index(drop=True)
)

print("Total sampled stocks:", len(sampled_universe))

sampled_universe.head(10)

Total sampled stocks: 100


,company_name,ticker,sector,isin,market_cap_lakh,market_cap_source,market_cap_rank,market_cap_stratum
0,Bharti Airtel Ltd.,BHARTIARTL,Telecommunication,INE397D01024,113625128.614847,"NSE 6M average market cap, Jul-Dec 2025",3,1
1,Infosys Ltd.,INFY,Information Technology,INE009A01021,63510354.939534,"NSE 6M average market cap, Jul-Dec 2025",7,1
2,Hindustan Unilever Ltd.,HINDUNILVR,Fast Moving Consumer Goods,INE030A01027,58251812.469492,"NSE 6M average market cap, Jul-Dec 2025",9,1
3,Larsen & Toubro Ltd.,LT,Construction,INE018A01030,51876327.592404,"NSE 6M average market cap, Jul-Dec 2025",11,1
4,UltraTech Cement Ltd.,ULTRACEMCO,Construction Materials,INE481G01011,35804548.314875,"NSE 6M average market cap, Jul-Dec 2025",19,1
5,Eternal Ltd.,ETERNAL,Consumer Services,INE758T01015,29713850.887578,"NSE 6M average market cap, Jul-Dec 2025",26,1
6,Bharat Electronics Ltd.,BEL,Capital Goods,INE263A01024,29301161.41487,"NSE 6M average market cap, Jul-Dec 2025",27,1
7,JSW Steel Ltd.,JSWSTEEL,Metals & Mining,INE019A01038,27041985.222946,"NSE 6M average market cap, Jul-Dec 2025",30,1
8,Adani Power Ltd.,ADANIPOWER,Power,INE814H01029,26436010.492891,"NSE 6M average market cap, Jul-Dec 2025",31,1
9,Asian Paints Ltd.,ASIANPAINT,Consumer Durables,INE021A01026,24642522.042464,"NSE 6M average market cap, Jul-Dec 2025",34,1


In [35]:
print("Stocks per market-cap stratum:")

print(
    sampled_universe[
        "market_cap_stratum"
    ]
    .value_counts()
    .sort_index()
)

print("\nStocks per sector:")

print(
    sampled_universe[
        "sector"
    ]
    .value_counts()
)

Stocks per market-cap stratum:
market_cap_stratum
1    20
2    20
3    20
4    20
5    20
Name: count, dtype: int64

Stocks per sector:
sector
Healthcare                           6
Financial Services                   6
Automobile and Auto Components       6
Capital Goods                        6
Construction                         5
Fast Moving Consumer Goods           5
Information Technology               5
Telecommunication                    5
Metals & Mining                      5
Power                                5
Consumer Services                    5
Construction Materials               5
Services                             5
Consumer Durables                    5
Chemicals                            5
Oil Gas & Consumable Fuels           5
Realty                               5
Textiles                             4
Media Entertainment & Publication    4
Diversified                          3
Name: count, dtype: int64


In [36]:
print(
    "Duplicate tickers:",
    sampled_universe["ticker"]
    .duplicated()
    .sum()
)

print(
    "Missing market caps:",
    sampled_universe["market_cap_lakh"]
    .isna()
    .sum()
)

Duplicate tickers: 0
Missing market caps: 0


In [37]:
sampled_universe["sampling_seed"] = RANDOM_SEED
sampled_universe["market_cap_period"] = "2025-07-01 to 2025-12-31"

sampled_universe[
    [
        "market_cap_rank",
        "company_name",
        "ticker",
        "sector",
        "market_cap_stratum",
        "market_cap_lakh"
    ]
].head(20)

,market_cap_rank,company_name,ticker,sector,market_cap_stratum,market_cap_lakh
0,3,Bharti Airtel Ltd.,BHARTIARTL,Telecommunication,1,113625128.614847
1,7,Infosys Ltd.,INFY,Information Technology,1,63510354.939534
2,9,Hindustan Unilever Ltd.,HINDUNILVR,Fast Moving Consumer Goods,1,58251812.469492
3,11,Larsen & Toubro Ltd.,LT,Construction,1,51876327.592404
4,19,UltraTech Cement Ltd.,ULTRACEMCO,Construction Materials,1,35804548.314875
5,26,Eternal Ltd.,ETERNAL,Consumer Services,1,29713850.887578
6,27,Bharat Electronics Ltd.,BEL,Capital Goods,1,29301161.41487
7,30,JSW Steel Ltd.,JSWSTEEL,Metals & Mining,1,27041985.222946
8,31,Adani Power Ltd.,ADANIPOWER,Power,1,26436010.492891
9,34,Asian Paints Ltd.,ASIANPAINT,Consumer Durables,1,24642522.042464


In [38]:
OUTPUT_FILE = "sampled_nifty500_universe_100.csv"

sampled_universe.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved:", OUTPUT_FILE)

Saved: sampled_nifty500_universe_100.csv


In [39]:
sector_benchmark_map = {
    "Automobile and Auto Components": "Nifty Auto",
    "Capital Goods": "Nifty Capital Goods",
    "Chemicals": "Nifty Chemicals",
    "Construction": "Nifty Construction",
    "Consumer Durables": "Nifty Consumer Durables",
    "Consumer Services": "Nifty Consumer Services",
    "Fast Moving Consumer Goods": "Nifty FMCG",
    "Financial Services": "Nifty Financial Services",
    "Healthcare": "Nifty Healthcare",
    "Information Technology": "Nifty IT",
    "Media Entertainment & Publication": "Nifty Media",
    "Metals & Mining": "Nifty Metal",
    "Oil Gas & Consumable Fuels": "Nifty Oil & Gas",
    "Power": "Nifty Power",
    "Realty": "Nifty Realty",
    "Telecommunication": "Nifty Telecommunications",

    # No sufficiently clean broad-sector benchmark for now
    "Construction Materials": None,
    "Services": None,
    "Diversified": None,
}

In [40]:
sampled_universe["sector_benchmark"] = (
    sampled_universe["sector"]
    .map(sector_benchmark_map)
)

sampled_universe["sector_benchmark_available"] = (
    sampled_universe["sector_benchmark"]
    .notna()
)

print("Benchmark available:")
print(
    sampled_universe[
        "sector_benchmark_available"
    ].value_counts()
)

print("\nStocks without sector benchmark:")

sampled_universe.loc[
    ~sampled_universe["sector_benchmark_available"],
    ["ticker", "company_name", "sector"]
]

Benchmark available:
sector_benchmark_available
True     83
False    17
Name: count, dtype: int64

Stocks without sector benchmark:


,ticker,company_name,sector
4,ULTRACEMCO,UltraTech Cement Ltd.,Construction Materials
11,INDIGO,InterGlobe Aviation Ltd.,Services
20,SHREECEM,Shree Cement Ltd.,Construction Materials
34,JSWINFRA,JSW Infrastructure Ltd.,Services
40,PAGEIND,Page Industries Ltd.,Textiles
46,GODREJIND,Godrej Industries Ltd.,Diversified
48,KPRMILL,K.P.R. Mill Ltd.,Textiles
49,3MINDIA,3M India Ltd.,Diversified
50,ACC,ACC Ltd.,Construction Materials
52,DELHIVERY,Delhivery Ltd.,Services


In [41]:
sector_benchmark_map["Construction Materials"] = "Nifty Cement"

sampled_universe["sector_benchmark"] = (
    sampled_universe["sector"]
    .map(sector_benchmark_map)
)

sampled_universe["sector_benchmark_available"] = (
    sampled_universe["sector_benchmark"]
    .notna()
)

print("Benchmark available:")
print(
    sampled_universe["sector_benchmark_available"]
    .value_counts()
)

print("\nStocks without sector benchmark:")

sampled_universe.loc[
    ~sampled_universe["sector_benchmark_available"],
    ["ticker", "company_name", "sector"]
]

Benchmark available:
sector_benchmark_available
True     88
False    12
Name: count, dtype: int64

Stocks without sector benchmark:


,ticker,company_name,sector
11,INDIGO,InterGlobe Aviation Ltd.,Services
34,JSWINFRA,JSW Infrastructure Ltd.,Services
40,PAGEIND,Page Industries Ltd.,Textiles
46,GODREJIND,Godrej Industries Ltd.,Diversified
48,KPRMILL,K.P.R. Mill Ltd.,Textiles
49,3MINDIA,3M India Ltd.,Diversified
52,DELHIVERY,Delhivery Ltd.,Services
67,ECLERX,eClerx Services Ltd.,Services
69,DCMSHRIRAM,DCM Shriram Ltd.,Diversified
82,TRIDENT,Trident Ltd.,Textiles


In [42]:
sampled_universe.to_csv(
    "sampled_nifty500_universe_100.csv",
    index=False
)

print("Final universe saved.")

Final universe saved.


## Interpretation

The final research universe contains 100 stocks sampled from the current Nifty 500.

The sample is deliberately constructed to provide:

- equal representation across five market-cap strata, with 20 stocks from each;
- broadly balanced representation across sectors;
- reproducible random selection using a fixed seed;
- official Nifty sector benchmarks wherever a defensible benchmark exists.

The resulting sample is intended for cross-sectional research into relative weakness across different company sizes and sectors.

It is not intended to reproduce the market-cap-weighted or sector-weighted composition of the Nifty 500.

## Final Universe Summary

- Parent universe: 500 current Nifty 500 constituents
- Final sample: 100 stocks
- Market-cap strata: 5
- Stocks per stratum: 20
- Unique sectors represented: 20
- Stocks with official sector benchmark: 88
- Stocks without a sufficiently clean sector benchmark: 12
- Sampling seed: 42
- Market-cap measurement period: July 1, 2025 to December 31, 2025

## Remaining Limitations

- Current Nifty 500 membership is used rather than historical constituent membership.
- This introduces survivorship bias into historical analysis.
- Sector balancing intentionally differs from the natural sector composition of the Nifty 500.
- Some broad NSE sectors do not have sufficiently precise official Nifty sector benchmarks and remain unmapped.
- Market-cap rankings use a July–December 2025 average rather than continuously changing current market capitalization.